# Driver Monitoring: geometric20-v2
Video-only, subject-disjoint training. Attach **thuannn18022005/uta-rldd-full** and the supplied code dataset.

This notebook trains a NEW checkpoint; it does not overwrite legacy models. Start with ALERT/DROWSY; LOW_VIGILANCE is optional. Read KAGGLE_20_FEATURES.md before running the extraction stage. No real-data results are included.

In [ ]:
import os, sys, subprocess, zipfile
from pathlib import Path
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
# A Kaggle code dataset may expose extracted files or the supplied archive.
candidates = list(INPUT.rglob('train20.py'))
if candidates:
    CODE = candidates[0].parents[1]
else:
    archives = list(INPUT.rglob('driver_monitoring_kaggle20.zip'))
    if not archives:
        raise FileNotFoundError('Attach the supplied driver_monitoring_kaggle20 code dataset')
    destination = WORK / 'code20'
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as z:
        for name in z.namelist():
            resolved = (destination / name).resolve()
            if not resolved.is_relative_to(destination.resolve()):
                raise ValueError('Invalid archive path')
        z.extractall(destination)
    CODE = destination / 'driver_monitoring'
os.chdir(CODE)
sys.path.insert(0, str(CODE))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(CODE/'requirements-kaggle20.txt'), '-q'], check=True)
import torch, numpy as np
print('Code:', CODE, 'PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())


In [ ]:
# Locate the original video dataset, including Kaggle's versioned mount layout.
folds = sorted(INPUT.rglob('Fold1_part1'))
if not folds:
    raise FileNotFoundError('Attach the full original RLDD video dataset; image-only mirrors do not work')
DATA_ROOT = folds[0].parent
if not list(DATA_ROOT.rglob('Fold5_part1')):
    raise ValueError('This mount has no fold 5. Select the full dataset.')
MANIFEST = WORK/'manifest20.csv'
MODEL = WORK/'face_landmarker.task'
CACHE = WORK/'cache20'
DATA = WORK/'processed20'
SAMPLE_FPS, SEQ_LEN, STRIDE = 15, 60, 15
INCLUDE_LOW = False
print('Verify dataset root:', DATA_ROOT)
print('Window:', SEQ_LEN/SAMPLE_FPS, 'seconds; labels:', '3 states' if INCLUDE_LOW else 'ALERT/DROWSY')


In [ ]:
import urllib.request, shutil
if not MODEL.exists() and (CODE/'models/face_landmarker.task').exists():
    shutil.copyfile(CODE/'models/face_landmarker.task', MODEL)
if not MODEL.exists():
    urllib.request.urlretrieve('https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task', MODEL)
from training.prepare20 import rldd_manifest
rldd_manifest(DATA_ROOT, MANIFEST, include_low=INCLUDE_LOW)
import csv
from collections import Counter
with MANIFEST.open() as f: rows = list(csv.DictReader(f))
print('Videos by split/label:', Counter((r['split'],r['label']) for r in rows))
for split in ('train','val','test'):
    print(split, 'subjects:', sorted({r['subject'] for r in rows if r['split']==split}))


## Extract once and preserve the cache
This is the expensive CPU/video-decoding stage. Do not delete cache20 on retries. Use the optional `training.prepare20 cache` shard commands in the guide to split extraction over sessions. If partial dataset output exists, use a new DATA directory while reusing CACHE. The original dataset is large; no full-video copy into /kaggle/working is needed.

In [ ]:
from training.prepare20 import build
if (DATA/'metadata.json').exists():
    print('Prepared data already exists:', DATA)
else:
    build(MANIFEST, MODEL, DATA, CACHE, sample_fps=SAMPLE_FPS, seq_len=SEQ_LEN, stride=STRIDE)
from training.train20 import load_data
splits, metadata = load_data(DATA)
for split, values in splits.items():
    print(split, values['X'].shape, np.bincount(values['y']))
del splits  # each training run loads its own arrays


## Compare using validation only
12 and 20 feature variants use identical windows, identities and labels. Train the small GRU first. Optionally run LSTM. A score on these windows is not an event-level alarm metric. Compare DROWSY recall as well as macro-F1, and repeat seeds before drawing conclusions.

In [ ]:
from training.train20 import train
for dim in (12, 20):
    output = WORK/f'gru{dim}_seed42'
    if output.exists():
        print('Existing experiment, keeping it:', output)
        continue
    train(DATA, output, feature_dim=dim, cell='gru', hidden=64, epochs=40, seed=42)


In [ ]:
import json
for run in sorted(WORK.glob('gru*_seed42')):
    if not (run/'history.json').exists(): continue
    history = json.loads((run/'history.json').read_text())
    best = max(history, key=lambda e: e['validation']['macro_f1'])
    print(run.name, 'epoch', best['epoch'], 'validation:', best['validation'])
# Optional: train(DATA, WORK/'lstm20_seed42', cell='lstm', epochs=40, seed=42)


## Final test, only after selecting the experiment
Choose the checkpoint based on validation, then set RUN_FINAL_TEST=True. Keep the test results out of subsequent tuning. The output includes probabilities and per-subject metrics; latency is model-only, not full camera FPS.

In [ ]:
from training.train20 import evaluate
RUN_FINAL_TEST = False
CHOSEN = WORK/'gru20_seed42'/'best.pt'  # change based on validation
if RUN_FINAL_TEST:
    evaluate(DATA, CHOSEN, WORK/'test_final')
else:
    print('Test held back. Select CHOSEN using validation, then enable final evaluation.')


## Save outputs
Preserve `manifest20.csv`, `processed20/metadata.json`, prepared splits, `cache20`, training run directories and final evaluation outputs. For inference, load with `models.temporal20.load_checkpoint` and the same `FeatureWindow` preprocessing. The legacy live_demo.py does not yet consume this schema.